# Preprocessing (v2)

Consolidated monolithic version.

In [ ]:
import os
import sys
import shutil
from pathlib import Path

# --- Environment Detection ---
IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''

# --- Setup Paths ---
if IS_KAGGLE:
    print("Umgebung: Kaggle")
    DATA_DIR = Path("/kaggle/input/datasets/axxtur/data-mining-2026-final-assignment")
    OUTPUTS_DIR = Path("/kaggle/working/outputs")
else:
    print("Umgebung: Lokal")
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    root = Path.cwd()
    for c in candidates:
        if (c / "data" / "test.csv").exists():
            root = c
            break
    DATA_DIR = root / "data"
    OUTPUTS_DIR = root / "outputs"

if IS_KAGGLE and not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input/data-mining-final-project")

TRAIN_PATH = DATA_DIR / "train.csv"
MODE = "full"
if not TRAIN_PATH.exists():
    TRAIN_PATH = DATA_DIR / "train_sample.csv"
    MODE = "sample"

TEST_PATH = DATA_DIR / "test.csv"
PROCESSED_DIR = OUTPUTS_DIR / "processed"
SUBMISSION_DIR = OUTPUTS_DIR / "submissions"
FIGURES_DIR = OUTPUTS_DIR / "figures"

for d in [PROCESSED_DIR, SUBMISSION_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

env = {
        "is_kaggle": IS_KAGGLE,
    "mode": MODE,
    "data_dir": DATA_DIR,
    "train_path": TRAIN_PATH,
    "test_path": TEST_PATH,
    "processed_dir": PROCESSED_DIR,
    "submission_dir": SUBMISSION_DIR,
    "outputs_dir": OUTPUTS_DIR,
    "figures_dir": FIGURES_DIR,
}

print(f"Modus: {MODE}")
print(f"Train Path: {TRAIN_PATH} ({'OK' if TRAIN_PATH.exists() else 'FEHLT'})")
print(f"Test Path:  {TEST_PATH} ({'OK' if TEST_PATH.exists() else 'FEHLT'})")


## Script: features_v2.py

In [ ]:
# --- NATIVE FEATURE ENGINEERING (v3) ---
import numpy as np
import pandas as pd

WEATHER_COLS = [
    "prec", "surf_pre", "humidity", "tmp", "dp_tmp",
    "wb_tmp", "tmp_max", "tmp_min", "tmp_range",
    "surf_tmp", "wind", "wind_max", "wind_min", "wind_range",
]

LAG_COLS = ["tmp_range", "tmp_max", "tmp", "prec", "wind", "surf_pre"]
LAGS = [1, 3, 7, 14, 21]

ROLL_COLS = ["prec", "wind", "tmp"]
ROLL_WINDOWS = [7, 14, 28]

def parse_dates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "year" not in df.columns:
        parts = df["date"].astype(str).str.split("-", expand=True)
        df["year"] = parts[0].astype(int)
        df["month"] = parts[1].astype(int)
        df["day"] = parts[2].astype(int)
    return df

def add_ordinal(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["ordinal"] = df["year"] * 372 + df["month"] * 31 + df["day"]
    return df

def sort_panel(df: pd.DataFrame) -> pd.DataFrame:
    if "ordinal" not in df.columns:
        df = add_ordinal(df)
    return df.sort_values(["region_id", "ordinal", "date"]).reset_index(drop=True)

def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["day_sin"] = np.sin(2 * np.pi * df["day"] / 31)
    df["day_cos"] = np.cos(2 * np.pi * df["day"] / 31)
    return df

def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    grouped = df.groupby("region_id", sort=False)
    for col in LAG_COLS:
        for lag in LAGS:
            df[f"{col}_lag{lag}"] = grouped[col].shift(lag)
    return df

def add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    grouped = df.groupby("region_id", sort=False)
    
    # 1. Standard rolling means/max
    for col in ROLL_COLS:
        prior = grouped[col].shift(1)
        for window in ROLL_WINDOWS:
            roller = prior.groupby(df["region_id"], sort=False)
            df[f"{col}_roll{window}_mean"] = roller.transform(lambda s: s.rolling(window, min_periods=3).mean())
            df[f"{col}_roll{window}_max"] = roller.transform(lambda s: s.rolling(window, min_periods=3).max())
            
            # Special: Accumulated Precipitation
            if col == "prec":
                df[f"prec_roll{window}_sum"] = roller.transform(lambda s: s.rolling(window, min_periods=3).sum())
    
    # 2. 90-day precipitation and temp anomaly
    prior_prec = grouped["prec"].shift(1)
    prior_tmp = grouped["tmp"].shift(1)
    
    roller_prec = prior_prec.groupby(df["region_id"], sort=False)
    roller_tmp = prior_tmp.groupby(df["region_id"], sort=False)
    
    df["prec_roll90_sum"] = roller_prec.transform(lambda s: s.rolling(90, min_periods=10).sum())
    df["tmp_roll90_mean"] = roller_tmp.transform(lambda s: s.rolling(90, min_periods=10).mean())
    df["tmp_mean_diff_90"] = df["tmp"] - df["tmp_roll90_mean"]
    
    # 3. Cross Features
    df["storm_proxy"] = df["wind_max"] * df["prec"]
    
    return df

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = parse_dates(df)
    df = add_ordinal(df)
    df = sort_panel(df)
    df = add_calendar_features(df)
    df = add_lag_features(df)
    df = add_rolling_features(df)
    return df

def feature_columns(include_region: bool = True) -> list[str]:
    lag_names = [f"{c}_lag{lag}" for c in LAG_COLS for lag in LAGS]
    roll_names = []
    for col in ROLL_COLS:
        for window in ROLL_WINDOWS:
            roll_names.extend([f"{col}_roll{window}_mean", f"{col}_roll{window}_max"])
            if col == "prec":
                roll_names.append(f"prec_roll{window}_sum")
                
    roll_names.extend(["prec_roll90_sum", "tmp_roll90_mean", "tmp_mean_diff_90", "storm_proxy"])
    
    calendar = ["month_sin", "month_cos", "day_sin", "day_cos"]
    cols = list(WEATHER_COLS) + lag_names + roll_names + calendar
    if include_region:
        cols = ["region_id"] + cols
    return cols

def add_persistence_baseline(df: pd.DataFrame, lag_days: int = 7) -> pd.DataFrame:
    df = df.copy()
    filled = df.groupby("region_id", sort=False)["score"].ffill()
    df["score_persist7"] = filled.groupby(df["region_id"], sort=False).shift(lag_days)
    return df

def build_features_v2(df: pd.DataFrame, region_stats: pd.DataFrame | None = None) -> pd.DataFrame:
    df = build_features(df)
    df = add_persistence_baseline(df, lag_days=7)
    return df

def feature_columns_v2(include_region: bool = True) -> list[str]:
    score_cols = ["score_persist7"]
    base = feature_columns(include_region=include_region)
    extra = score_cols
    return list(dict.fromkeys(base + extra))

def meta_train_cols_v2() -> list[str]:
    return [
        "region_id",
        "date",
        "year",
        "month",
        "day",
        "ordinal",
        "score",
        "score_persist7",
    ]

def save_columns_v2(df: pd.DataFrame, *, labeled: bool) -> list[str]:
    meta = meta_train_cols_v2() if labeled else [
        "region_id",
        "date",
        "year",
        "month",
        "day",
        "ordinal",
    ]
    feats = feature_columns_v2()
    return [c for c in list(dict.fromkeys(meta + feats)) if c in df.columns]


## Script: parallel_util.py

In [ ]:
"""Shared parallelism helpers for preprocessing and modeling."""
from __future__ import annotations

import os
from concurrent.futures import FIRST_COMPLETED, ProcessPoolExecutor, wait
from typing import Callable, Iterator, TypeVar

T = TypeVar("T")
R = TypeVar("R")


def default_workers(cap: int = 8) -> int:
    """
    Worker count: env ``DM_WORKERS``, else ``CPU-1`` (capped).

    Set ``DM_WORKERS=1`` to disable multiprocessing.
    """
    env = os.environ.get("DM_WORKERS", "").strip()
    if env:
        return max(1, int(env))
    import multiprocessing as mp

    return max(1, min(cap, mp.cpu_count() - 1))


def run_parallel_map(
    fn: Callable[[T], R],
    tasks: list[T],
    *,
    n_workers: int | None = None,
) -> list[R]:
    """``pool.map`` — order preserved; fine for small task lists (e.g. 5 weeks)."""
    if not tasks:
        return []
    n_workers = n_workers or default_workers()
    if n_workers <= 1:
        return [fn(t) for t in tasks]
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        return list(pool.map(fn, tasks))


def run_parallel_consume(
    fn: Callable[[T], R],
    tasks: Iterator[T],
    consume: Callable[[R], None],
    *,
    n_workers: int | None = None,
    max_inflight: int | None = None,
) -> int:
    """
    Stream tasks through a process pool; call *consume* on each result as ready.
    Returns number of tasks completed.
    """
    n_workers = n_workers or default_workers()
    max_inflight = max_inflight or n_workers * 2
    n_done = 0

    if n_workers <= 1:
        for task in tasks:
            consume(fn(task))
            n_done += 1
        return n_done

    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        inflight: dict = {}
        task_iter = iter(tasks)

        def submit_one() -> bool:
            try:
                task = next(task_iter)
            except StopIteration:
                return False
            inflight[pool.submit(fn, task)] = True
            return True

        for _ in range(max_inflight):
            if not submit_one():
                break

        while inflight:
            done, _ = wait(inflight, return_when=FIRST_COMPLETED)
            for fut in done:
                inflight.pop(fut)
                consume(fut.result())
                n_done += 1
                submit_one()
    return n_done


## Script: preprocess_streaming_v2.py

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

class _ParquetAppender:
    def __init__(self, path) -> None:
        self.path = path
        self._writer = None

    def write(self, df) -> None:
        if df.empty:
            return
        table = pa.Table.from_pandas(df, preserve_index=False)
        if self._writer is None:
            self.path.parent.mkdir(parents=True, exist_ok=True)
            self._writer = pq.ParquetWriter(self.path, table.schema)
        self._writer.write_table(table)

    def close(self) -> None:
        if self._writer is not None:
            self._writer.close()
            self._writer = None

"""
Preprocessing v2 — streaming + optional multi-core per region.

Outputs:
  train_labeled_v2.parquet
  test_features_v2.parquet
"""
from __future__ import annotations

from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq





OUT_TRAIN_V2 = "train_labeled_v2.parquet"
OUT_TEST_V2 = "test_features_v2.parquet"


def process_region_v2_core(
    train_part: pd.DataFrame,
    test_part: pd.DataFrame,
    region_stats: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Feature engineering for one region (no I/O)."""
    train_part = train_part.copy()
    test_part = test_part.copy()
    train_part["_split"] = "train"
    test_part["_split"] = "test"
    if "score" not in test_part.columns:
        test_part["score"] = float("nan")

    panel = pd.concat([train_part, test_part], ignore_index=True)
    panel = build_features_v2(panel, region_stats=region_stats)

    train_feat = panel[panel["_split"] == "train"]
    test_feat = panel[panel["_split"] == "test"]
    test_raw = parse_dates(test_part.drop(columns=["_split"], errors="ignore"))

    train_labeled = train_feat[train_feat["score"].notna()]
    return (
        train_labeled[save_columns_v2(train_labeled, labeled=True)],
        test_feat[save_columns_v2(test_feat, labeled=False)],
    )


def _region_worker_v2(
    args: tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_r, test_r, region_stats = args
    return process_region_v2_core(train_r, test_r, region_stats)


def _process_region_v2(
    train_part: pd.DataFrame,
    test_part: pd.DataFrame,
    region_stats: pd.DataFrame,
    train_writer: _ParquetAppender,
    test_writer: _ParquetAppender,
) -> None:
    train_out, test_out = process_region_v2_core(train_part, test_part, region_stats)
    train_writer.write(train_out)
    test_writer.write(test_out)


def _iter_region_tasks(
    train_path: Path,
    test_by_region: dict,
    region_stats: pd.DataFrame,
    chunk_size: int,
):
    """Yield (train_r, test_r, region_stats) per completed region."""
    buffer_parts: list[pd.DataFrame] = []
    current_region = None

    for chunk in pd.read_csv(train_path, chunksize=chunk_size):
        chunk = parse_dates(chunk)
        for region, g in chunk.groupby("region_id", sort=False):
            if current_region is None:
                current_region = region
                buffer_parts = [g]
                continue

            if region == current_region:
                buffer_parts.append(g)
                continue

            train_r = pd.concat(buffer_parts, ignore_index=True)
            test_r = test_by_region.get(current_region, pd.DataFrame())
            yield (train_r, test_r, region_stats)
            current_region = region
            buffer_parts = [g]

    if current_region is not None and buffer_parts:
        train_r = pd.concat(buffer_parts, ignore_index=True)
        test_r = test_by_region.get(current_region, pd.DataFrame())
        yield (train_r, test_r, region_stats)


def preprocess_by_region_v2(
    train_path: Path,
    test_path: Path,
    out_train: Path,
    out_test: Path,
    chunk_size: int = 500_000,
    n_workers: int | None = None,
) -> dict:
    n_workers = n_workers if n_workers is not None else default_workers()
    
    print("Berechne region_stats vorab...")
    train_scores = pd.read_csv(train_path, usecols=["region_id", "score"])
    labeled = train_scores[train_scores["score"].notna()]
    region_stats = (
        labeled.groupby("region_id", sort=False)["score"]
        .agg(mean="mean", median="median", std="std")
        .rename(
            columns={
                "mean": "region_score_mean",
                "median": "region_score_median",
                "std": "region_score_std",
            }
        )
        .reset_index()
    )
    region_stats["region_score_std"] = region_stats["region_score_std"].fillna(0.0)
    del train_scores, labeled
    print(f"  -> {len(region_stats)} Regionen berechnet.")

    test = parse_dates(pd.read_csv(test_path))
    test_by_region = {r: g for r, g in test.groupby("region_id", sort=False)}

    for path in (out_train, out_test):
        if path.exists():
            path.unlink()

    train_writer = _ParquetAppender(out_train)
    test_writer = _ParquetAppender(out_test)
    finished_regions: set[str] = set()
    duplicate_test_skipped = 0

    def _consume(result: tuple[pd.DataFrame, pd.DataFrame]) -> None:
        nonlocal duplicate_test_skipped
        train_out, test_out = result
        if train_out.empty and test_out.empty:
            return

        rid = str(
            test_out["region_id"].iloc[0]
            if not test_out.empty
            else train_out["region_id"].iloc[0]
        )

        if rid in finished_regions:
            duplicate_test_skipped += 1
            if not train_out.empty:
                train_writer.write(train_out)
            return

        finished_regions.add(rid)
        if not train_out.empty:
            train_writer.write(train_out)
        if not test_out.empty:
            test_writer.write(test_out)

        if len(finished_regions) % 200 == 0:
            print(f"  … {len(finished_regions)} Regionen verarbeitet")

    try:
        tasks = _iter_region_tasks(train_path, test_by_region, region_stats, chunk_size)
        run_parallel_consume(
            _region_worker_v2,
            tasks,
            _consume,
            n_workers=n_workers,
        )
    finally:
        train_writer.close()
        test_writer.close()

    train_rows = pq.read_metadata(out_train).num_rows if out_train.exists() else 0
    test_rows = pq.read_metadata(out_test).num_rows if out_test.exists() else 0

    if duplicate_test_skipped:
        print(
            f"  Hinweis: {duplicate_test_skipped} doppelte Region-Durchläufe "
            "(test nur 1× geschrieben)."
        )

    return {
        "version": 2,
        "regions": len(finished_regions),
        "duplicate_region_passes": duplicate_test_skipped,
        "train_labeled_rows": train_rows,
        "test_rows": test_rows,
        "out_train": out_train,
        "out_test": out_test,
        "feature_count": len(feature_columns_v2()),
        "n_workers": n_workers,
    }


# 03b – Preprocessing v2 (erweiterte Features)

**Vergleich zu v1:** [docs/10_PREPROCESSING_V2.md](../docs/10_PREPROCESSING_V2.md) · Original: `03_preprocessing.ipynb`

| Output v2 | v1 (alt) |
|-----------|----------|
| `train_labeled_v2.parquet` | `train_labeled.parquet` |
| `test_features_v2.parquet` | `test_features.parquet` |

Neu: Score-Lags, `score_persist7` als Feature, Regional-Score-Stats, 91-Tage-Test-Aggregate.

**Parallel:** mehrere Regionen gleichzeitig (`DM_WORKERS`, Standard ≈ CPU−1). Kaggle-Tipp: CSV nach `/content/` kopieren (Zelle 2).

→ danach **`04b_modeling_v2.ipynb`**

In [ ]:
import os
import shutil


path_train_v2 = PROCESSED_DIR / "train_labeled_v2.parquet"
path_test_v2 = PROCESSED_DIR / "test_features_v2.parquet"
N_WORKERS = default_workers()

# Kaggle: /kaggle/input I/O vermeiden (oft schneller als Paid-CPU)
train_path = TRAIN_PATH
test_path = TEST_PATH
if env.get("is_kaggle"):
    local_train = Path("/kaggle/working/train.csv") if env.get("is_kaggle") else Path("/content/train.csv")
    local_test = Path("/kaggle/working/test.csv") if env.get("is_kaggle") else Path("/content/test.csv")
    if not local_train.exists():
        print("Kopiere train.csv → /content/ …")
        shutil.copy(TRAIN_PATH, local_train)
    if not local_test.exists():
        print("Kopiere test.csv → /content/ …")
        shutil.copy(TEST_PATH, local_test)
    train_path, test_path = local_train, local_test


if MODE == "full":
    print(f"Streaming v2 parallel ({N_WORKERS} workers) …")
    stats = preprocess_by_region_v2(
        train_path, test_path, path_train_v2, path_test_v2,
        chunk_size=500_000, n_workers=N_WORKERS,
    )
    print("Fertig:", stats)
else:
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    train_labeled, test_feat = preprocess_panel_v2(train, test, n_workers=N_WORKERS)
    train_labeled.to_parquet(path_train_v2, index=False)
    test_feat.to_parquet(path_test_v2, index=False)
    print(f"Gespeichert: {len(train_labeled):,} train | {len(test_feat):,} test")

In [ ]:
import pyarrow.parquet as pq

n_train = pq.read_metadata(path_train_v2).num_rows
n_test = pq.read_metadata(path_test_v2).num_rows
print(f"Parquet v2: train {n_train:,} | test {n_test:,}")

head = pd.read_parquet(path_train_v2)
v2_only = [c for c in head.columns if c.startswith(("score_lag", "region_score", "test91_"))]
print(f"Neue Spalten (Beispiel): {v2_only[:8]} … ({len(v2_only)} v2-spezifisch sichtbar)")
head.head(3)